In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# Load clean data
X = pd.read_csv('../datasets/heart_X_clean.csv')
y = pd.read_csv('../datasets/heart_y_clean.csv').squeeze()

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size: {X_train.shape}")
print(f"Test size: {X_test.shape}")

Train size: (242, 13)
Test size: (61, 13)


In [2]:
# Model 1: Logistic Regression
lr = LogisticRegression(random_state=42)
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

# Model 2: Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

# Model 3: XGBoost
from xgboost import XGBClassifier
xgb = XGBClassifier(random_state=42, eval_metric='logloss')
xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_test)

print("All 3 models trained!")

All 3 models trained!


In [4]:
# Store all results in a clean table
results = {}

for name, model, pred in [
    ('Logistic Regression', lr, lr_pred),
    ('Random Forest', rf, rf_pred),
    ('XGBoost', xgb, xgb_pred)
]:
    results[name] = {
        'Accuracy': round(accuracy_score(y_test, pred), 4),
        'F1 Score': round(f1_score(y_test, pred, average='weighted'), 4),
        'ROC-AUC': round(roc_auc_score(y_test, model.predict_proba(X_test), multi_class='ovr'), 4)
    }

# Print as table
results_df = pd.DataFrame(results).T
print("\nResults on Heart Disease Dataset:")
print(results_df)



Results on Heart Disease Dataset:
                     Accuracy  F1 Score  ROC-AUC
Logistic Regression    0.5574    0.5450   0.7635
Random Forest          0.5246    0.4759   0.7647
XGBoost                0.4754    0.4808   0.7515


In [5]:
# Cross-validation gives more reliable results than single split
print("\n5-Fold Cross-Validation Results:")
print("-" * 45)

for name, model in [
    ('Logistic Regression', lr),
    ('Random Forest', rf),
    ('XGBoost', xgb)
]:
    cv_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
    print(f"{name}:")
    print(f"  Mean Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")


5-Fold Cross-Validation Results:
---------------------------------------------
Logistic Regression:
  Mean Accuracy: 0.5778 ± 0.0728
Random Forest:
  Mean Accuracy: 0.5807 ± 0.0288
XGBoost:
  Mean Accuracy: 0.5741 ± 0.0286


In [7]:
# Save results table
results_df.to_csv('../results/heart_results.csv')

print("Results saved!")
print("\nYour first real research results:")
print(results_df)

Results saved!

Your first real research results:
                     Accuracy  F1 Score  ROC-AUC
Logistic Regression    0.5574    0.5450   0.7635
Random Forest          0.5246    0.4759   0.7647
XGBoost                0.4754    0.4808   0.7515
